# Lora 实战

## Step1 导入相关包

In [34]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [35]:
import shutil
import os

# 源路径（input目录下的文件/文件夹）
source_path = "/kaggle/input/alpaca-data-zh/alpaca_data_zh/"

# 目标路径（working目录下，可自定义名称）
target_path = "/kaggle/working/alpaca_data_zh/"

# 若目标路径已存在，先删除（可选）
if os.path.exists(target_path):
    shutil.rmtree(target_path)  # 删除文件夹
    # 若复制单个文件，用 os.remove(target_path)

# 复制文件夹（包括所有子文件）
shutil.copytree(source_path, target_path)

# 若复制单个文件，用：
# shutil.copyfile(source_file, target_file)

# 验证是否复制成功
print("复制后的文件列表：", os.listdir(target_path))


ds = Dataset.load_from_disk("/kaggle/working/alpaca_data_zh/")
ds

复制后的文件列表： ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']


Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [36]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

In [37]:
from huggingface_hub import snapshot_download

# 仓库地址（如 "Langboat/bloom-1b4-zh"）
repo_id = "Qwen/Qwen3-0.6B"
# 下载路径（自定义，如 "./bloom-1b4-zh"）
local_dir = "./Qwen/Qwen3-0.6B"

# 下载整个仓库
snapshot_download(
    repo_id=repo_id,
    local_dir=local_dir,
    # 可选：指定分支（默认 main）
    revision="main",
    # 可选：强制覆盖已有文件
    force_download=True
)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

'/kaggle/working/Qwen/Qwen3-0.6B'

In [38]:
!pwd

/kaggle/working


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step3 数据集预处理

In [39]:
tokenizer = AutoTokenizer.from_pretrained("./Qwen/Qwen3-0.6B")
tokenizer

Qwen2TokenizerFast(name_or_path='./Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normaliz

In [40]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [41]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Map:   0%|          | 0/26858 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [42]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。<|im_end|>'

In [43]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。<|im_end|>'

## Step4 创建模型

In [44]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

In [45]:
for name, parameter in model.named_parameters():
    print(name)

model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.v_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.self_attn.q_norm.weight
model.layers.0.self_attn.k_norm.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.v_proj.weight
model.layers.1.self_attn.o_proj.weight
model.layers.1.self_attn.q_norm.weight
model.layers.1.self_attn.k_norm.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.self_attn.q_proj.weight
model.layers.2.self_attn.k_proj.weight
model.layers.2.self_attn.v_proj.weight
model.l

## Lora

### PEFT Step1 配置文件

In [46]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM, modules_to_save=["word_embeddings"])
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules=None, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=['word_embeddings'], init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

### PEFT Step2 创建模型

In [47]:
model = get_peft_model(model, config)

In [48]:
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='Qwen/Qwen3-0.6B', revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=['word_embeddings'], init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

In [49]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.model.embed_tokens.weight
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.weight
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.weight
base_model.model.model.layers.0.self_attn.q_norm.weight
base_model.model.model.layers.0.self_attn.k_norm.weight
base_model.model.model.layers.0.mlp.gate_proj.weight
base_model.model.model.layers.0.mlp.up_proj.weight
base_model.model.model.layers.0.mlp.down_proj.weight
base_model.model.model.layers.0.input_layernorm.weight
base_model.model.model.layers.0.post_attention_layernorm.weight
base_model.model.model.layers.1.self_at

In [50]:
model.print_trainable_parameters()

trainable params: 1,146,880 || all params: 597,196,800 || trainable%: 0.1920


## Step5 配置训练参数

In [51]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=2,
    report_to="none"
)

## Step6 创建训练器

In [52]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

/tmp/ipykernel_38/1988846643.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## Step7 模型训练

In [ ]:
trainer.train()

Step,Training Loss
10,1.295400
20,1.273600
30,1.115500
40,1.198900
50,1.181500
60,1.085500
70,1.044700
80,1.024100
90,0.986600
100,1.079800


## Step8 模型推理

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("你叫什么名字？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("你知道江西科技学院吗？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("马上要考研了，我很紧张怎么办呀？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("马上要考研了，我很紧张怎么办呀？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True)[0], skip_special_tokens=True)

In [63]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("最近晚上总是失眠，很烦人怎么办？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True)[0], skip_special_tokens=True)

'Human: 最近晚上总是失眠，很烦人怎么办？\n\nAssistant: 失眠可能有很多种原因，包括压力、焦虑、抑郁、睡眠呼吸暂停、药物副作用等。如果你感到很烦人，可以尝试一些放松技巧，如深呼吸、冥想、正念练习、听音乐、散步、泡澡等。也可以尝试调整作息时间，保证充足的睡眠。如果你愿意的话，可以咨询医生，寻求专业的睡眠问题的帮助。'

In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("最近晚上总是失眠，很烦人怎么办？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=False)[0], skip_special_tokens=True)

In [64]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("我想要举办一次党团日活动，你推荐一些什么活动", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=False)[0], skip_special_tokens=True)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Human: 我想要举办一次党团日活动，你推荐一些什么活动\n\nAssistant: 党团日活动可以围绕党的理论、政策、历史和现实进行，也可以围绕团的建设、学习和实践等方面展开。以下是一些推荐的活动：\n\n1. 党团知识讲座：邀请专家学者或团组织成员，围绕党的理论、政策、历史和现实进行讲解和分享。\n\n2. 党团活动：组织参观红色教育基地、红色纪念馆、红色文化展览等，让参与者深入了解党的历史和精神。\n\n3. 党团实践：组织参观团建活动、志愿服务、公益活动等，让参与者在实践中学习和锻炼。\n\n4. 党团学习：组织学习党的理论、政策和团的建设，通过阅读书籍、参加讲座、讨论等方式，加深对党的认识和理解。\n\n5. 党团互动：组织团建活动，通过游戏、竞赛、交流等方式，增进团成员之间的感情和友谊。\n\n6. 党团文艺演出：组织文艺演出，通过音乐、舞蹈、诗歌朗诵等方式，丰富团日活动内容，增强团日活动的趣味性。\n\n7. 党团志愿服务：组织志愿者服务活动，通过帮助他人、服务社区等方式，增强团成员的奉献精神和责任感。\n\n8. 党团摄影：组织摄影比赛，通过拍摄团日活动中的精彩瞬间，记录团日活动的精彩瞬间，增强团日活动的纪念意义。\n\n9. 党团美食：组织团日活动中的美食节，通过品尝团日活动中的美食，增强团日活动的趣味性。\n\n10. 党团游戏：组织团日活动中的游戏，通过游戏、竞赛、互动等方式，增强团日活动的趣味性。'

In [65]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("我想要举办一次党团日活动，你推荐一些什么活动", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True)[0], skip_special_tokens=True)

'Human: 我想要举办一次党团日活动，你推荐一些什么活动\n\nAssistant: 党团日活动通常以增强党组织凝聚力、加强党员教育为核心，通常包括以下一些活动：\n\n1. 纪念活动：参观红色教育基地，参观革命遗址，观看革命电影，观看历史影像资料，重温入党誓词等。\n\n2. 重温入党誓词：通过重温入党誓词，让党员进一步坚定理想信念，牢记党的宗旨，增强党的凝聚力。\n\n3. 专题学习：通过学习党的理论知识，党的历史，党的方针政策，党的宗旨和要求，让党员深入学习党的知识，提高政治素养。\n\n4. 政治活动：参加团组织的活动，如理论学习会，主题讨论会，集体讨论会，组织学习，组织交流等，增强团组织的凝聚力。\n\n5. 互动活动：组织党员和团员参加各类活动，如志愿服务，公益活动，社会实践等，增强党组织的凝聚力和号召力。\n\n6. 团队建设：组织团组织内部的各项活动，如团建活动，团队竞赛，团队合作，团队建设等，增强团组织的凝聚力和战斗力。\n\n7. 竞赛活动：组织团组织内部的各类竞赛活动，如演讲比赛，知识竞赛，文艺演出等，增强团组织的凝聚力和活力。\n\n总之，党团日活动应围绕增强党组织凝聚力和战斗力，加强党员教育，提高政治素养，促进团组织建设，丰富团组织活动内容，增强团组织的凝聚力和战斗力。'